In [1]:
import pandas as pd
import re
import emoji
from langdetect import detect, DetectorFactory

In [2]:
# 1. Đọc dữ liệu từ file CSV
# Đảm bảo file 'ket_qua_tong_hop.csv' nằm cùng thư mục với đoạn code này
df = pd.read_csv(r'D:\User\Tài liệu học\HK2 - 2025-2026\1. Data\raw_data.csv')

In [3]:
# 2. Xử lý dữ liệu ban đầu

# Xử lý Username bị thiếu theo số thứ tự dòng
df['Username'] = df.apply(
    lambda row: f"Anonymous_{row.name}" if pd.isna(row['Username']) else row['Username'],
    axis=1
)

# Tạo review_id dạng số cho mỗi dòng review gốc
df = df.reset_index(drop=True)
df['review_id'] = range(1, len(df) + 1)

# =========================
# Làm sạch sơ bộ cột Comment
# =========================

# Chuyển về string để xử lý an toàn
df['Comment'] = df['Comment'].astype(str)

# Xóa khoảng trắng thừa đầu/cuối
df['Comment'] = df['Comment'].str.strip()

# Danh sách các giá trị xem như rỗng / vô nghĩa
invalid_values = {
    '', 'na', 'n/a', 'null', 'none', 'nan', '//', '/', '-', '--', '---'
}

# Xóa các dòng Comment bị null/rỗng/giả null
df = df[
    ~df['Comment'].str.lower().isin(invalid_values)
].copy()

print(f"Số dòng sau khi xóa comment null/na/rỗng: {len(df)}")

Số dòng sau khi xóa comment null/na/rỗng: 68505


In [4]:
# Cố định seed để kết quả nhận diện không bị thay đổi ngẫu nhiên giữa các lần chạy
DetectorFactory.seed = 42

# Hàm nhận diện ngôn ngữ an toàn
def detect_language(text):
    try:
        text = str(text).strip()
        # Nếu bình luận quá ngắn (< 3 ký tự) hoặc toàn emoji/số sẽ rất khó nhận diện
        if len(text) < 3:
            return 'unknown'
        return detect(text)
    except:
        # Bắt lỗi với các dòng chỉ chứa link, số, hoặc ký tự đặc biệt
        return 'unknown'

print("⏳ Đang nhận diện ngôn ngữ từng bình luận... (Quá trình này mất vài phút tùy lượng dữ liệu)")

# 1. Tạo cột 'language' và áp dụng hàm nhận diện
df['language'] = df['Comment'].apply(detect_language)

# 2. Đếm số lượng của từng ngôn ngữ (không tính các bình luận 'unknown')
lang_counts = df[df['language'] != 'unknown']['language'].value_counts()

# 3. Lấy ra danh sách 10 ngôn ngữ có số lượng nhiều nhất
top_10_langs = lang_counts.head(10).index.tolist()

print("\n🏆 Top 10 ngôn ngữ có nhiều bình luận nhất:")
for lang, count in lang_counts.head(10).items():
    print(f"  - {lang.upper()}: {count} bình luận")

# 4. Lọc DataFrame, CHỈ giữ lại các dòng thuộc top 10 ngôn ngữ này
df = df[df['language'].isin(top_10_langs)].copy()

print(f"\n✅ Số lượng bình luận còn lại sau khi lọc giữ top 10: {len(df)}")

⏳ Đang nhận diện ngôn ngữ từng bình luận... (Quá trình này mất vài phút tùy lượng dữ liệu)

🏆 Top 10 ngôn ngữ có nhiều bình luận nhất:
  - EN: 49472 bình luận
  - VI: 8353 bình luận
  - KO: 2313 bình luận
  - FR: 1399 bình luận
  - DE: 1214 bình luận
  - JA: 1012 bình luận
  - RU: 832 bình luận
  - ES: 717 bình luận
  - IT: 538 bình luận
  - NL: 449 bình luận

✅ Số lượng bình luận còn lại sau khi lọc giữ top 10: 66299


In [5]:
# 3. Hàm làm sạch comment (KHÔNG tách câu nữa)

def clean_comment(text):
    if pd.isna(text):
        return None

    text = str(text).strip()
    if text == "":
        return None

    # Xóa xuống dòng
    text = text.replace("\n", " ").replace("\r", " ")

    # Xóa emoji / icon
    text = emoji.replace_emoji(text, replace='')

    # Chuẩn hóa khoảng trắng
    text = re.sub(r'\s+', ' ', text).strip()

    # Đưa về dạng thường để check
    text_lower = text.lower()

    # Các câu vô nghĩa / không phải review thật
    meaningless_phrases = {
        'automated translate',
        'automatic translation',
        'translated by google',
        'google translate',
        'auto translate',
        'machine translated',
        'test'
    }

    # Chỉ toàn dấu câu / ký tự đặc biệt
    if re.fullmatch(r'[\W_]+', text_lower):
        return None

    # Các câu kiểu auto translate...
    if text_lower in meaningless_phrases:
        return None

    # Quá ngắn: ít hơn 2 từ
    word_count = len(text_lower.split())
    if word_count < 3:
        return None

    return text

In [6]:
# 4. Áp dụng hàm xử lý lên toàn bộ dữ liệu
print("Đang xử lý dữ liệu, vui lòng đợi...")

# Làm sạch comment
df['Comment_cleaned'] = df['Comment'].apply(clean_comment)

before_drop_invalid = len(df)

# Xóa các dòng comment không đạt yêu cầu sau làm sạch
df = df[df['Comment_cleaned'].notna()].copy()

after_drop_invalid = len(df)
dropped_meaningless_comments = before_drop_invalid - after_drop_invalid

# =========================
# Xóa duplicate bình thường của cả dòng
# =========================
before_drop_duplicate = len(df)

df = df.drop_duplicates().copy()

after_drop_duplicate = len(df)
dropped_duplicates = before_drop_duplicate - after_drop_duplicate

# Tạo dữ liệu cuối
processed_data = []

for _, row in df.iterrows():
    processed_data.append({
        'review_id': row['review_id'],
        'Username': row['Username'],
        'Address': row['Address'],
        'Rating': row['Rating'],
        'Language': row['language'],
        'Comment': row['Comment_cleaned']
    })

Đang xử lý dữ liệu, vui lòng đợi...


In [7]:
# 5. Tạo DataFrame mới
final_df = pd.DataFrame(processed_data)

# ==========================================
# PHẦN BÁO CÁO KẾT QUẢ
# ==========================================
print("\n" + "="*40)
print("📊 BÁO CÁO XỬ LÝ DỮ LIỆU:")
print("="*40)
print(f"🔹 Số dòng sau khi lọc top 10 ngôn ngữ: {before_drop_invalid:,}")
print(f"🗑️ Số comment vô nghĩa / quá ngắn bị xóa: {dropped_meaningless_comments:,}")
print(f"🗑️ Số comment duplicate bị xóa: {dropped_duplicates:,}")
print(f"✅ Số dòng cuối cùng còn lại: {len(final_df):,}")
print("="*40 + "\n")

# Xóa cột phụ nếu cần xem lại df
df = df.drop(columns=['Comment_dedup_key'], errors='ignore')

# Lưu file CSV mới
file_output = 'processed data.csv'
final_df.to_csv(file_output, index=False, encoding='utf-8-sig')

print(f"Hoàn tất! File đã được lưu với tên '{file_output}'")
print(final_df.head())


📊 BÁO CÁO XỬ LÝ DỮ LIỆU:
🔹 Số dòng sau khi lọc top 10 ngôn ngữ: 66,299
🗑️ Số comment vô nghĩa / quá ngắn bị xóa: 1,892
🗑️ Số comment duplicate bị xóa: 0
✅ Số dòng cuối cùng còn lại: 64,407

Hoàn tất! File đã được lưu với tên 'processed data.csv'
   review_id     Username        Address  Rating Language  \
0          1      Mashajo  An Bang Beach       5       ru   
1          2         cj h  An Bang Beach       5       en   
2          3  Hoàng Quỳnh  An Bang Beach       5       vi   
3          4  Snus Hoi An  An Bang Beach       5       en   
4          5    Lavinie A  An Bang Beach       5       en   

                                             Comment  
0  Он находится примерно в 10-15 минутах езды от ...  
1              Quiet and quaint. Very laid back vive  
2  Đây là một bãi biển thư giãn tuyệt vời với làn...  
3  While spending time in Hoi An, I was looking f...  
4  Hi everyone—October’s storm caused severe dama...  
